# This notebook aim to find the sweet spot for cleaning process.

#Block 1 — Install, import, configuration

In [ ]:
!pip install -q kagglehub imagehash scikit-image tqdm scikit-learn torchvision matplotlib seaborn

In [ ]:
import os
import cv2
import json
import glob
import time
import warnings
from pathlib import Path
from itertools import combinations
from collections import defaultdict

import imagehash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CONFIG = {
    "dataset": "tongpython/cat-and-dog",
    "workspace": "/kaggle/working/sweet_spot_threshold" if Path("/kaggle/working").exists() else "/content/sweet_spot_threshold",
    "image_extensions": ["*.jpg", "*.jpeg", "*.png"],
    "proxy": {
        "backbone": "efficientnet_b0",
        "image_size": 224,
        "batch_size": 128,
        "num_workers": 0,
        "val_size": 0.2,
        "classifier_C": 0.1,
    },
    "dedup": {
        "default_hamming_threshold": 4,
        "lsh_bands": 8,
    },
    "score": {
        "retention_penalty": 0.10,
        "imbalance_penalty": 0.05,
    }
}

WORKSPACE = Path(CONFIG["workspace"])
RESULT_DIR = WORKSPACE / "results"
FIG_DIR = RESULT_DIR / "figures"
FEATURE_DIR = WORKSPACE / "features"

for d in [WORKSPACE, RESULT_DIR, FIG_DIR, FEATURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    print("Saved:", path)

print("Device:", DEVICE)
print("Workspace:", WORKSPACE)

#Block 2 — Dataset discovery and utility functions

In [ ]:
def get_dataset_root():
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        found = []
        for ext in CONFIG["image_extensions"]:
            found.extend(kaggle_input.rglob(ext))
        if found:
            return kaggle_input

    import kagglehub
    path = kagglehub.dataset_download(CONFIG["dataset"])
    return Path(path)


DATA_ROOT = get_dataset_root()
print("Dataset root:", DATA_ROOT)


def list_image_paths(root):
    paths = []
    for ext in CONFIG["image_extensions"]:
        paths.extend(root.rglob(ext))
    return sorted([str(p) for p in paths])


def infer_label(path):
    p = Path(path)
    name = p.name.lower()
    parent = p.parent.name.lower()
    parts = [x.lower() for x in p.parts]

    if parent in ["cat", "cats"] or name.startswith("cat"):
        return 0, "Cat"
    if parent in ["dog", "dogs"] or name.startswith("dog"):
        return 1, "Dog"

    if "cats" in parts or "cat" in parts:
        return 0, "Cat"
    if "dogs" in parts or "dog" in parts:
        return 1, "Dog"

    return None, None


all_paths = list_image_paths(DATA_ROOT)
print("Total image files found:", len(all_paths))

label_preview = []
for p in all_paths[:20]:
    label, label_name = infer_label(p)
    label_preview.append((Path(p).name, Path(p).parent.name, label_name))

display(pd.DataFrame(label_preview, columns=["file", "parent", "inferred_label"]).head(10))

In [ ]:
#Block 3 — Metric functions

In [ ]:
def image_entropy(gray):
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256]).flatten()
    hist = hist[hist > 0]
    if hist.sum() == 0:
        return 0.0
    p = hist / hist.sum()
    return float(-np.sum(p * np.log2(p)))


def blur_laplacian(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def gray_tolerance_metrics(rgb):
    arr = rgb.astype(np.float32)
    channel_range = arr.max(axis=2) - arr.min(axis=2)
    return {
        "gray_diff_mean": float(channel_range.mean()),
        "gray_diff_p95": float(np.percentile(channel_range, 95)),
        "near_gray_ratio_10": float((channel_range <= 10).mean()),
    }


def near_mono_metrics(gray):
    return {
        "dark_ratio": float((gray < 15).mean()),
        "bright_ratio": float((gray > 240).mean()),
        "near_mono_ratio": float(((gray < 15) | (gray > 240)).mean()),
    }


def saturation_metrics(bgr):
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    s = hsv[:, :, 1].astype(np.float32)
    return {
        "mean_sat": float(s.mean()),
        "std_sat": float(s.std()),
    }


def chromaticity_metrics(rgb):
    arr = rgb.astype(np.float32)
    chroma = arr.max(axis=2) - arr.min(axis=2)

    f = arr / 255.0
    R, G, B = f[:, :, 0], f[:, :, 1], f[:, :, 2]

    X = 0.4124 * R + 0.3576 * G + 0.1805 * B
    Y = 0.2126 * R + 0.7152 * G + 0.0722 * B
    Z = 0.0193 * R + 0.1192 * G + 0.9505 * B
    denom = X + Y + Z

    mask = denom > 1e-3
    if mask.sum() < 100:
        xy_spread = 0.0
    else:
        x = X[mask] / denom[mask]
        y = Y[mask] / denom[mask]
        xy_spread = float(x.std() + y.std())

    return {
        "chroma_mean": float(chroma.mean()),
        "chroma_std": float(chroma.std()),
        "chromaticity_spread": xy_spread,
    }


def saliency_centering_metrics(gray):
    small = cv2.resize(gray, (128, 128), interpolation=cv2.INTER_AREA)
    gx = cv2.Sobel(small, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(small, cv2.CV_32F, 0, 1, ksize=3)
    sal = np.sqrt(gx * gx + gy * gy)

    total = float(sal.sum()) + 1e-6
    h, w = sal.shape
    y0, y1 = h // 4, 3 * h // 4
    x0, x1 = w // 4, 3 * w // 4

    center_mass = float(sal[y0:y1, x0:x1].sum())
    return {
        "center_saliency_ratio": center_mass / total,
        "saliency_mean": float(sal.mean()),
        "saliency_max": float(sal.max()),
    }


def compression_artifact_score(gray):
    h, w = gray.shape
    if h < 16 or w < 16:
        return 0.0

    g = gray.astype(np.float32)
    gx = np.abs(np.diff(g, axis=1))
    gy = np.abs(np.diff(g, axis=0))

    block_cols = np.arange(7, w - 1, 8)
    block_rows = np.arange(7, h - 1, 8)

    if len(block_cols) == 0 or len(block_rows) == 0:
        return 0.0

    edge_x = gx[:, block_cols].mean()
    edge_y = gy[block_rows, :].mean()
    global_grad = (gx.mean() + gy.mean()) / 2.0 + 1e-6

    return float(((edge_x + edge_y) / 2.0) / global_grad)


def compute_phash_hex(pil_img):
    return str(imagehash.phash(pil_img, hash_size=8))


def hamming_hex(h1, h2):
    return (int(h1, 16) ^ int(h2, 16)).bit_count()


def inspect_image(path):
    label, label_name = infer_label(path)
    if label is None:
        return None

    bgr = cv2.imread(path)
    if bgr is None:
        return {
            "path": path,
            "label": label,
            "label_name": label_name,
            "is_corrupted": True,
        }

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape

    try:
        pil_img = Image.open(path).convert("RGB")
        phash = compute_phash_hex(pil_img)
    except Exception:
        phash = None

    rec = {
        "path": path,
        "label": label,
        "label_name": label_name,
        "is_corrupted": False,
        "width": int(w),
        "height": int(h),
        "min_side": int(min(w, h)),
        "max_side": int(max(w, h)),
        "aspect_extremity": float(max(w / h, h / w)),
        "mean_brightness": float(gray.mean()),
        "brightness_std": float(gray.std()),
        "blur_laplacian": blur_laplacian(gray),
        "entropy": image_entropy(gray),
        "compression_artifact": compression_artifact_score(gray),
        "phash": phash,
    }

    rec.update(gray_tolerance_metrics(rgb))
    rec.update(near_mono_metrics(gray))
    rec.update(saturation_metrics(bgr))
    rec.update(chromaticity_metrics(rgb))
    rec.update(saliency_centering_metrics(gray))

    return rec

#Block 4 — Audit dataset and save metrics

In [ ]:
AUDIT_PATH = RESULT_DIR / "audit_metrics.csv"

if AUDIT_PATH.exists():
    audit_df = pd.read_csv(AUDIT_PATH)
    print("Loaded audit cache:", AUDIT_PATH)
else:
    records = []
    for p in tqdm(all_paths, desc="Auditing images"):
        try:
            rec = inspect_image(p)
            if rec is not None:
                records.append(rec)
        except Exception:
            label, label_name = infer_label(p)
            if label is not None:
                records.append({
                    "path": p,
                    "label": label,
                    "label_name": label_name,
                    "is_corrupted": True,
                })

    audit_df = pd.DataFrame(records)
    audit_df = audit_df.reset_index(drop=True)
    audit_df.to_csv(AUDIT_PATH, index=False)
    print("Saved audit cache:", AUDIT_PATH)

print("Audit shape:", audit_df.shape)
display(audit_df.head())

metric_cols = [
    "width", "height", "min_side", "aspect_extremity",
    "blur_laplacian", "gray_diff_mean", "near_gray_ratio_10",
    "dark_ratio", "bright_ratio", "near_mono_ratio",
    "entropy", "mean_sat", "chroma_mean", "chromaticity_spread",
    "center_saliency_ratio", "compression_artifact", "brightness_std",
]

display(audit_df[metric_cols].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95, 0.99]).T.round(4))
display(audit_df["label_name"].value_counts().to_frame("count"))

#Block 5 — pHash near-duplicate detection

In [ ]:
def duplicate_mask_lsh(df, hamming_threshold=4, bands=8):
    valid = df["phash"].notna().values
    hashes = df["phash"].fillna("0").astype(str).tolist()
    n = len(df)

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    buckets = defaultdict(list)
    bits_per_band = 64 // bands

    for i, h in enumerate(hashes):
        if not valid[i]:
            continue
        value = int(h, 16)
        for b in range(bands):
            shift = b * bits_per_band
            band_value = (value >> shift) & ((1 << bits_per_band) - 1)
            buckets[(b, band_value)].append(i)

    candidate_pairs = set()
    for bucket in buckets.values():
        if len(bucket) < 2:
            continue
        for a, b in combinations(bucket, 2):
            if a < b:
                candidate_pairs.add((a, b))
            else:
                candidate_pairs.add((b, a))

    for a, b in tqdm(candidate_pairs, desc=f"Verifying pHash pairs ≤ {hamming_threshold}"):
        if hamming_hex(hashes[a], hashes[b]) <= hamming_threshold:
            union(a, b)

    clusters = np.array([find(i) for i in range(n)])

    q = (
        df["blur_laplacian"].rank(pct=True).fillna(0)
        + df["entropy"].rank(pct=True).fillna(0)
        + df["min_side"].rank(pct=True).fillna(0)
        + df["brightness_std"].rank(pct=True).fillna(0)
        - df["compression_artifact"].rank(pct=True).fillna(0)
    )

    keep = np.zeros(n, dtype=bool)
    temp = pd.DataFrame({
        "idx": np.arange(n),
        "cluster": clusters,
        "quality": q.values,
    })

    for _, g in temp.groupby("cluster"):
        best_idx = g.sort_values("quality", ascending=False)["idx"].iloc[0]
        keep[best_idx] = True

    duplicate = ~keep
    return duplicate, clusters


DEDUP_SUMMARY_PATH = RESULT_DIR / "dedup_summary.csv"
DEDUP_DEFAULT = CONFIG["dedup"]["default_hamming_threshold"]

dedup_records = []
for thr in [0, 2, 4, 6, 8]:
    dup_mask, clusters = duplicate_mask_lsh(
        audit_df,
        hamming_threshold=thr,
        bands=CONFIG["dedup"]["lsh_bands"],
    )

    dedup_records.append({
        "hamming_threshold": thr,
        "duplicates": int(dup_mask.sum()),
        "duplicate_pct": 100 * float(dup_mask.mean()),
        "kept": int((~dup_mask).sum()),
        "kept_pct": 100 * float((~dup_mask).mean()),
        "n_clusters": int(pd.Series(clusters).nunique()),
    })

    if thr == DEDUP_DEFAULT:
        audit_df["dup_cluster"] = clusters
        audit_df["is_duplicate"] = dup_mask

dedup_df = pd.DataFrame(dedup_records)
dedup_df.to_csv(DEDUP_SUMMARY_PATH, index=False)

display(dedup_df)

audit_df.to_csv(AUDIT_PATH, index=False)
print("Updated audit file with duplicate columns:", AUDIT_PATH)

#Block 6 — Metric distributions and correlations

In [ ]:
plot_metrics = [
    ("blur_laplacian", "Blur: Laplacian Variance"),
    ("gray_diff_mean", "Gray Tolerance: Mean RGB Channel Difference"),
    ("near_mono_ratio", "Near-Monochrome Ratio"),
    ("dark_ratio", "Dark Pixel Ratio"),
    ("bright_ratio", "Bright Pixel Ratio"),
    ("aspect_extremity", "Aspect Ratio Extremity"),
    ("min_side", "Minimum Resolution"),
    ("entropy", "Image Entropy"),
    ("mean_sat", "Mean Saturation"),
    ("chroma_mean", "Chromaticity: Mean Channel Range"),
    ("center_saliency_ratio", "Center Saliency Ratio"),
    ("compression_artifact", "Compression Artifact Score"),
    ("brightness_std", "Brightness Standard Deviation"),
]

fig, axes = plt.subplots(5, 3, figsize=(18, 22))
axes = axes.flatten()

for ax, (col, title) in zip(axes, plot_metrics):
    sns.kdeplot(
        data=audit_df,
        x=col,
        hue="label_name",
        fill=True,
        alpha=0.18,
        common_norm=False,
        ax=ax,
    )
    ax.set_title(title, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.35)

for ax in axes[len(plot_metrics):]:
    ax.axis("off")

plt.suptitle("Quality Metric Distributions by Class", fontsize=16, fontweight="bold")
savefig("01_quality_metric_distributions.png")
plt.show()


corr_cols = [c for c, _ in plot_metrics]
corr = audit_df[corr_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    annot_kws={"size": 7},
)
plt.title("Correlation Matrix of Image Quality Metrics", fontweight="bold")
savefig("02_quality_metric_correlation.png")
plt.show()

#Block 7 — Visual inspection helpers

In [ ]:
def show_metric_extremes(df, metric, ascending=True, n=12, title=None):
    sub = df.dropna(subset=[metric]).sort_values(metric, ascending=ascending).head(n)
    if len(sub) == 0:
        print("No images to show.")
        return

    cols = min(6, n)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.0, rows * 3.2))

    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sub.iterrows()):
        try:
            img = Image.open(row["path"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "Error", ha="center", va="center")
        ax.axis("off")
        ax.set_title(
            f'{row["label_name"]}\n{metric}={row[metric]:.3g}',
            fontsize=8,
        )

    for ax in axes[len(sub):]:
        ax.axis("off")

    plt.suptitle(title or f"Extreme samples: {metric}", fontsize=13, fontweight="bold")
    safe_name = f"extreme_{metric}_{'low' if ascending else 'high'}.png"
    savefig(safe_name)
    plt.show()


show_metric_extremes(audit_df, "blur_laplacian", ascending=True, title="Lowest Blur Scores")
show_metric_extremes(audit_df, "min_side", ascending=True, title="Smallest Images")
show_metric_extremes(audit_df, "entropy", ascending=True, title="Lowest Entropy Images")
show_metric_extremes(audit_df, "near_mono_ratio", ascending=False, title="Highest Near-Monochrome Ratio")
show_metric_extremes(audit_df, "compression_artifact", ascending=False, title="Highest Compression Artifact Scores")
show_metric_extremes(audit_df, "center_saliency_ratio", ascending=True, title="Lowest Center Saliency Ratio")

#Block 8 — Extract proxy features once

In [ ]:
PROXY_TRANSFORM = T.Compose([
    T.Resize((CONFIG["proxy"]["image_size"], CONFIG["proxy"]["image_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class ProxyImageDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), (128, 128, 128))
        return self.transform(img)


def build_proxy_backbone():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model = nn.Sequential(model.features, nn.AdaptiveAvgPool2d((1, 1)))
    model = model.to(DEVICE).eval()
    return model


@torch.inference_mode()
def extract_proxy_features(paths):
    dataset = ProxyImageDataset(paths, PROXY_TRANSFORM)
    loader = DataLoader(
        dataset,
        batch_size=CONFIG["proxy"]["batch_size"],
        shuffle=False,
        num_workers=CONFIG["proxy"]["num_workers"],
        pin_memory=(DEVICE.type == "cuda"),
    )

    model = build_proxy_backbone()
    features = []

    for imgs in tqdm(loader, desc="Extracting EfficientNet-B0 features"):
        out = model(imgs.to(DEVICE)).view(imgs.size(0), -1)
        features.append(out.cpu().numpy())

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return np.vstack(features)


FEATURE_PATH = FEATURE_DIR / "X_proxy_efficientnet_b0.npy"
LABEL_PATH = FEATURE_DIR / "y_proxy.npy"

if FEATURE_PATH.exists() and LABEL_PATH.exists():
    X_all = np.load(FEATURE_PATH)
    y_all = np.load(LABEL_PATH)
    if len(X_all) != len(audit_df):
        X_all = extract_proxy_features(audit_df["path"].tolist())
        y_all = audit_df["label"].values
        np.save(FEATURE_PATH, X_all)
        np.save(LABEL_PATH, y_all)
else:
    X_all = extract_proxy_features(audit_df["path"].tolist())
    y_all = audit_df["label"].values
    np.save(FEATURE_PATH, X_all)
    np.save(LABEL_PATH, y_all)

print("Feature matrix:", X_all.shape)
print("Labels:", y_all.shape)

#Block 9 — Fixed proxy validation split

In [ ]:
base_eval_mask = ~audit_df["is_corrupted"].fillna(False)

if "is_duplicate" in audit_df.columns:
    base_eval_mask = base_eval_mask & ~audit_df["is_duplicate"].fillna(False)

base_indices = audit_df[base_eval_mask].index.to_numpy()
base_labels = audit_df.loc[base_indices, "label"].values

train_idx, val_idx = train_test_split(
    base_indices,
    test_size=CONFIG["proxy"]["val_size"],
    stratify=base_labels,
    random_state=SEED,
)

raw_dog_ratio = audit_df["label"].mean()

print("Proxy train size:", len(train_idx))
print("Proxy validation size:", len(val_idx))
print("Train class distribution:")
display(audit_df.loc[train_idx, "label_name"].value_counts().to_frame("count"))
print("Validation class distribution:")
display(audit_df.loc[val_idx, "label_name"].value_counts().to_frame("count"))


def evaluate_cleaning_mask(mask, name, threshold):
    mask = pd.Series(mask, index=audit_df.index).fillna(False)

    train_keep_idx = np.array([i for i in train_idx if mask.loc[i]])
    val_keep_idx = val_idx

    if len(train_keep_idx) < 100:
        return {
            "filter": name,
            "threshold": threshold,
            "accuracy": np.nan,
            "f1_macro": np.nan,
            "precision_macro": np.nan,
            "recall_macro": np.nan,
            "n_train": len(train_keep_idx),
            "n_val": len(val_keep_idx),
            "retention_pct": 100 * mask.mean(),
            "train_retention_pct": 100 * len(train_keep_idx) / len(train_idx),
            "dog_ratio_after": np.nan,
            "imbalance_shift": np.nan,
            "score": np.nan,
        }

    y_train = y_all[train_keep_idx]
    if len(np.unique(y_train)) < 2:
        return {
            "filter": name,
            "threshold": threshold,
            "accuracy": np.nan,
            "f1_macro": np.nan,
            "precision_macro": np.nan,
            "recall_macro": np.nan,
            "n_train": len(train_keep_idx),
            "n_val": len(val_keep_idx),
            "retention_pct": 100 * mask.mean(),
            "train_retention_pct": 100 * len(train_keep_idx) / len(train_idx),
            "dog_ratio_after": np.nan,
            "imbalance_shift": np.nan,
            "score": np.nan,
        }

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=CONFIG["proxy"]["classifier_C"],
            max_iter=1000,
            random_state=SEED,
            n_jobs=-1,
        ))
    ])

    clf.fit(X_all[train_keep_idx], y_all[train_keep_idx])
    preds = clf.predict(X_all[val_keep_idx])

    acc = accuracy_score(y_all[val_keep_idx], preds)
    f1 = f1_score(y_all[val_keep_idx], preds, average="macro")
    prec = precision_score(y_all[val_keep_idx], preds, average="macro")
    rec = recall_score(y_all[val_keep_idx], preds, average="macro")

    kept_df = audit_df[mask]
    dog_ratio_after = kept_df["label"].mean() if len(kept_df) else np.nan
    imbalance_shift = abs(dog_ratio_after - raw_dog_ratio) if len(kept_df) else np.nan
    retention_frac = mask.mean()

    score = (
        f1
        - CONFIG["score"]["retention_penalty"] * (1 - retention_frac)
        - CONFIG["score"]["imbalance_penalty"] * imbalance_shift
    )

    return {
        "filter": name,
        "threshold": threshold,
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": prec,
        "recall_macro": rec,
        "n_train": len(train_keep_idx),
        "n_val": len(val_keep_idx),
        "retention_pct": 100 * retention_frac,
        "train_retention_pct": 100 * len(train_keep_idx) / len(train_idx),
        "dog_ratio_after": dog_ratio_after,
        "imbalance_shift": imbalance_shift,
        "score": score,
    }


baseline_mask = base_eval_mask.copy()
baseline_result = evaluate_cleaning_mask(baseline_mask, "baseline", "none")
display(pd.DataFrame([baseline_result]).round(4))

#Block 10 — Single-metric threshold sweep

In [ ]:
def m_blur(t):
    return base_eval_mask & (audit_df["blur_laplacian"] >= t)

def m_gray(t):
    return base_eval_mask & (audit_df["gray_diff_mean"] >= t)

def m_dark(t):
    return base_eval_mask & (audit_df["dark_ratio"] <= t)

def m_bright(t):
    return base_eval_mask & (audit_df["bright_ratio"] <= t)

def m_near_mono(t):
    return base_eval_mask & (audit_df["near_mono_ratio"] <= t)

def m_aspect(t):
    return base_eval_mask & (audit_df["aspect_extremity"] <= t)

def m_min_res(t):
    return base_eval_mask & (audit_df["min_side"] >= t)

def m_entropy_min(t):
    return base_eval_mask & (audit_df["entropy"] >= t)

def m_entropy_max(t):
    return base_eval_mask & (audit_df["entropy"] <= t)

def m_sat_min(t):
    return base_eval_mask & (audit_df["mean_sat"] >= t)

def m_sat_max(t):
    return base_eval_mask & (audit_df["mean_sat"] <= t)

def m_chroma(t):
    return base_eval_mask & (audit_df["chroma_mean"] >= t)

def m_saliency(t):
    return base_eval_mask & (audit_df["center_saliency_ratio"] >= t)

def m_compression(t):
    return base_eval_mask & (audit_df["compression_artifact"] <= t)

def m_brightness_min(t):
    return base_eval_mask & (audit_df["brightness_std"] >= t)

def m_brightness_max(t):
    return base_eval_mask & (audit_df["brightness_std"] <= t)


FILTER_SPECS = [
    ("blur_laplacian_min", [20, 30, 50, 75, 100], m_blur),
    ("gray_diff_mean_min", [0, 1, 2, 5, 10, 15], m_gray),
    ("dark_ratio_max", [0.90, 0.95, 0.98, 0.99], m_dark),
    ("bright_ratio_max", [0.90, 0.95, 0.98, 0.99], m_bright),
    ("near_mono_ratio_max", [0.90, 0.95, 0.98, 0.99], m_near_mono),
    ("aspect_extremity_max", [2.5, 3.0, 4.0, 5.0, 8.0], m_aspect),
    ("min_side_min", [32, 48, 64, 96, 128], m_min_res),
    ("entropy_min", [2.0, 3.0, 4.0, 5.0, 5.5, 6.0], m_entropy_min),
    ("entropy_max", [7.0, 7.3, 7.5, 7.8, 8.0], m_entropy_max),
    ("mean_sat_min", [0, 5, 10, 20, 30, 50], m_sat_min),
    ("mean_sat_max", [180, 200, 220, 240, 255], m_sat_max),
    ("chroma_mean_min", [0, 2, 5, 10, 15, 20], m_chroma),
    ("center_saliency_ratio_min", [0.10, 0.15, 0.20, 0.25, 0.30], m_saliency),
    ("compression_artifact_max", [1.2, 1.5, 2.0, 2.5, 3.0, 5.0], m_compression),
    ("brightness_std_min", [5, 10, 15, 20, 30, 40], m_brightness_min),
    ("brightness_std_max", [70, 85, 100, 120, 160, 255], m_brightness_max),
]

SWEEP_PATH = RESULT_DIR / "single_metric_sweep.csv"

if SWEEP_PATH.exists():
    sweep_df = pd.read_csv(SWEEP_PATH)
    print("Loaded sweep cache:", SWEEP_PATH)
else:
    records = [baseline_result]

    for name, thresholds, mask_fn in FILTER_SPECS:
        print(f"\nSweeping {name}")
        for t in thresholds:
            mask = mask_fn(t)
            result = evaluate_cleaning_mask(mask, name, t)
            records.append(result)
            print(
                f"{name}={t} | "
                f"F1={result['f1_macro']:.4f} | "
                f"Acc={result['accuracy']:.4f} | "
                f"Retention={result['retention_pct']:.1f}% | "
                f"Score={result['score']:.4f}"
            )

    sweep_df = pd.DataFrame(records)
    sweep_df.to_csv(SWEEP_PATH, index=False)
    print("Saved:", SWEEP_PATH)

display(sweep_df.sort_values("score", ascending=False).head(20).round(4))

#Block 11 — Sweep visualization and best thresholds

In [ ]:
best_single_df = (
    sweep_df[sweep_df["filter"] != "baseline"]
    .dropna(subset=["score"])
    .sort_values("score", ascending=False)
    .groupby("filter", as_index=False)
    .head(1)
    .sort_values("score", ascending=False)
)

display(best_single_df.round(4))

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=sweep_df[sweep_df["filter"] != "baseline"],
    x="retention_pct",
    y="f1_macro",
    hue="filter",
    s=90,
)
plt.axhline(baseline_result["f1_macro"], linestyle="--", color="black", label="Baseline F1")
plt.title("Single-Metric Threshold Sweep: Macro F1 vs Retention", fontweight="bold")
plt.xlabel("Retention (%)")
plt.ylabel("Validation Macro F1")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="both", linestyle="--", alpha=0.3)
savefig("03_single_metric_f1_vs_retention.png")
plt.show()


plt.figure(figsize=(12, 7))
sns.barplot(
    data=best_single_df,
    x="score",
    y="filter",
)
plt.title("Best Threshold per Metric by Penalized Score", fontweight="bold")
plt.xlabel("Penalized score")
plt.ylabel("Metric")
savefig("04_best_single_thresholds.png")
plt.show()

#Block 12 — Combined preset evaluation

In [ ]:
def build_cleaning_mask(df, cfg):
    mask = ~df["is_corrupted"].fillna(False)

    if cfg.get("remove_duplicates", True) and "is_duplicate" in df.columns:
        mask = mask & ~df["is_duplicate"].fillna(False)

    if cfg.get("blur_laplacian_min") is not None:
        mask = mask & (df["blur_laplacian"] >= cfg["blur_laplacian_min"])

    if cfg.get("min_side_min") is not None:
        mask = mask & (df["min_side"] >= cfg["min_side_min"])

    if cfg.get("aspect_extremity_max") is not None:
        mask = mask & (df["aspect_extremity"] <= cfg["aspect_extremity_max"])

    if cfg.get("dark_ratio_max") is not None:
        dark_bad = (df["dark_ratio"] > cfg["dark_ratio_max"]) & (df["entropy"] < cfg.get("entropy_near_mono_max", 3.0))
        mask = mask & ~dark_bad

    if cfg.get("bright_ratio_max") is not None:
        bright_bad = (df["bright_ratio"] > cfg["bright_ratio_max"]) & (df["entropy"] < cfg.get("entropy_near_mono_max", 3.0))
        mask = mask & ~bright_bad

    soft_flags = pd.Series(0, index=df.index, dtype=int)

    if cfg.get("gray_diff_mean_min") is not None:
        soft_flags += (df["gray_diff_mean"] < cfg["gray_diff_mean_min"]).astype(int)

    if cfg.get("entropy_min") is not None:
        soft_flags += (df["entropy"] < cfg["entropy_min"]).astype(int)

    if cfg.get("entropy_max") is not None:
        soft_flags += (df["entropy"] > cfg["entropy_max"]).astype(int)

    if cfg.get("mean_sat_min") is not None:
        soft_flags += (df["mean_sat"] < cfg["mean_sat_min"]).astype(int)

    if cfg.get("mean_sat_max") is not None:
        soft_flags += (df["mean_sat"] > cfg["mean_sat_max"]).astype(int)

    if cfg.get("chroma_mean_min") is not None:
        soft_flags += (df["chroma_mean"] < cfg["chroma_mean_min"]).astype(int)

    if cfg.get("center_saliency_ratio_min") is not None:
        soft_flags += (df["center_saliency_ratio"] < cfg["center_saliency_ratio_min"]).astype(int)

    if cfg.get("compression_artifact_max") is not None:
        soft_flags += (df["compression_artifact"] > cfg["compression_artifact_max"]).astype(int)

    if cfg.get("brightness_std_min") is not None:
        soft_flags += (df["brightness_std"] < cfg["brightness_std_min"]).astype(int)

    if cfg.get("brightness_std_max") is not None:
        soft_flags += (df["brightness_std"] > cfg["brightness_std_max"]).astype(int)

    max_soft_flags = cfg.get("max_soft_flags", 2)
    mask = mask & (soft_flags <= max_soft_flags)

    return mask, soft_flags


CLEANING_PRESETS = {
    "none_valid_only": {
        "remove_duplicates": False,
        "max_soft_flags": 99,
    },
    "conservative": {
        "remove_duplicates": True,
        "blur_laplacian_min": 20,
        "min_side_min": 32,
        "aspect_extremity_max": 8.0,
        "dark_ratio_max": 0.99,
        "bright_ratio_max": 0.99,
        "entropy_near_mono_max": 2.0,
        "gray_diff_mean_min": 0,
        "entropy_min": 2.0,
        "entropy_max": 8.0,
        "mean_sat_min": 0,
        "mean_sat_max": 255,
        "chroma_mean_min": 0,
        "center_saliency_ratio_min": 0.10,
        "compression_artifact_max": 5.0,
        "brightness_std_min": 5,
        "brightness_std_max": 255,
        "max_soft_flags": 3,
    },
    "balanced": {
        "remove_duplicates": True,
        "blur_laplacian_min": 50,
        "min_side_min": 64,
        "aspect_extremity_max": 5.0,
        "dark_ratio_max": 0.95,
        "bright_ratio_max": 0.95,
        "entropy_near_mono_max": 3.0,
        "gray_diff_mean_min": 1,
        "entropy_min": 3.0,
        "entropy_max": 7.8,
        "mean_sat_min": 5,
        "mean_sat_max": 240,
        "chroma_mean_min": 2,
        "center_saliency_ratio_min": 0.15,
        "compression_artifact_max": 3.0,
        "brightness_std_min": 10,
        "brightness_std_max": 160,
        "max_soft_flags": 2,
    },
    "strict": {
        "remove_duplicates": True,
        "blur_laplacian_min": 75,
        "min_side_min": 96,
        "aspect_extremity_max": 4.0,
        "dark_ratio_max": 0.95,
        "bright_ratio_max": 0.95,
        "entropy_near_mono_max": 4.0,
        "gray_diff_mean_min": 2,
        "entropy_min": 4.0,
        "entropy_max": 7.5,
        "mean_sat_min": 10,
        "mean_sat_max": 230,
        "chroma_mean_min": 5,
        "center_saliency_ratio_min": 0.20,
        "compression_artifact_max": 2.5,
        "brightness_std_min": 15,
        "brightness_std_max": 120,
        "max_soft_flags": 1,
    },
}


PRESET_PATH = RESULT_DIR / "cleaning_preset_results.csv"

preset_records = []
soft_flag_tables = {}

for preset_name, preset_cfg in CLEANING_PRESETS.items():
    mask, soft_flags = build_cleaning_mask(audit_df, preset_cfg)
    result = evaluate_cleaning_mask(mask, preset_name, "preset")
    preset_records.append(result)
    soft_flag_tables[preset_name] = soft_flags

preset_df = pd.DataFrame(preset_records)
preset_df.to_csv(PRESET_PATH, index=False)

display(preset_df.sort_values("score", ascending=False).round(4))

plt.figure(figsize=(9, 5))
sns.barplot(
    data=preset_df.sort_values("score", ascending=False),
    x="filter",
    y="f1_macro",
)
plt.axhline(baseline_result["f1_macro"], linestyle="--", color="black", label="Baseline")
plt.title("Cleaning Preset Comparison", fontweight="bold")
plt.xlabel("Cleaning preset")
plt.ylabel("Validation Macro F1")
plt.xticks(rotation=20, ha="right")
plt.legend()
savefig("05_cleaning_preset_comparison.png")
plt.show()

#Block 13 — Choose final cleaning configuration

In [ ]:
best_preset_row = preset_df.sort_values("score", ascending=False).iloc[0]
best_preset_name = best_preset_row["filter"]
best_config = CLEANING_PRESETS[best_preset_name]

final_mask, final_soft_flags = build_cleaning_mask(audit_df, best_config)
final_clean_df = audit_df[final_mask].copy()
removed_df = audit_df[~final_mask].copy()

print("Selected preset:", best_preset_name)
display(pd.DataFrame([best_preset_row]).round(4))

summary = pd.DataFrame({
    "group": ["before_cleaning", "after_cleaning", "removed"],
    "total": [len(audit_df), len(final_clean_df), len(removed_df)],
    "cat": [
        int((audit_df["label"] == 0).sum()),
        int((final_clean_df["label"] == 0).sum()),
        int((removed_df["label"] == 0).sum()),
    ],
    "dog": [
        int((audit_df["label"] == 1).sum()),
        int((final_clean_df["label"] == 1).sum()),
        int((removed_df["label"] == 1).sum()),
    ],
})

summary["retention_pct"] = [
    100.0,
    100 * len(final_clean_df) / len(audit_df),
    100 * len(removed_df) / len(audit_df),
]

display(summary.round(2))

plt.figure(figsize=(8, 5))
plot_df = summary.melt(id_vars="group", value_vars=["cat", "dog"], var_name="class", value_name="count")
sns.barplot(data=plot_df, x="group", y="count", hue="class")
plt.title("Class Counts Before and After Cleaning", fontweight="bold")
plt.xlabel("")
plt.ylabel("Number of images")
savefig("06_before_after_cleaning_class_counts.png")
plt.show()


final_clean_path = RESULT_DIR / "final_clean_images.csv"
final_config_path = RESULT_DIR / "recommended_cleaning_config.json"

final_clean_df[["path", "label", "label_name"]].to_csv(final_clean_path, index=False)

final_report = {
    "version": "sweet_spot_threshold_final",
    "selected_preset": best_preset_name,
    "dataset": CONFIG["dataset"],
    "proxy_model": "EfficientNet-B0 fixed feature extractor + Logistic Regression",
    "decision_rule": {
        "hard_filters": [
            "corrupted image",
            "duplicate image",
            "blur_laplacian_min",
            "min_side_min",
            "aspect_extremity_max",
            "near-mono dark/bright with entropy condition",
        ],
        "soft_flags": [
            "gray_diff_mean_min",
            "entropy_min",
            "entropy_max",
            "mean_sat_min",
            "mean_sat_max",
            "chroma_mean_min",
            "center_saliency_ratio_min",
            "compression_artifact_max",
            "brightness_std_min",
            "brightness_std_max",
        ],
        "max_soft_flags": best_config.get("max_soft_flags", None),
    },
    "recommended_config": best_config,
    "results": {
        "baseline_f1_macro": float(baseline_result["f1_macro"]),
        "selected_f1_macro": float(best_preset_row["f1_macro"]),
        "selected_accuracy": float(best_preset_row["accuracy"]),
        "selected_score": float(best_preset_row["score"]),
        "retention_pct": float(best_preset_row["retention_pct"]),
        "train_retention_pct": float(best_preset_row["train_retention_pct"]),
        "n_total": int(len(audit_df)),
        "n_clean": int(len(final_clean_df)),
        "n_removed": int(len(removed_df)),
        "cat_clean": int((final_clean_df["label"] == 0).sum()),
        "dog_clean": int((final_clean_df["label"] == 1).sum()),
    }
}

with open(final_config_path, "w") as f:
    json.dump(final_report, f, indent=4)

print("Saved clean image list:", final_clean_path)
print("Saved recommended config:", final_config_path)
print(json.dumps(final_report, indent=4)[:2000])


#Block 14 — Removed-image reason analysis

In [ ]:
def removal_reasons(row, cfg):
    reasons = []

    if row.get("is_corrupted", False):
        reasons.append("corrupted")

    if cfg.get("remove_duplicates", True) and row.get("is_duplicate", False):
        reasons.append("duplicate")

    if cfg.get("blur_laplacian_min") is not None and row["blur_laplacian"] < cfg["blur_laplacian_min"]:
        reasons.append("blurry")

    if cfg.get("min_side_min") is not None and row["min_side"] < cfg["min_side_min"]:
        reasons.append("undersized")

    if cfg.get("aspect_extremity_max") is not None and row["aspect_extremity"] > cfg["aspect_extremity_max"]:
        reasons.append("extreme_aspect_ratio")

    if cfg.get("dark_ratio_max") is not None:
        if row["dark_ratio"] > cfg["dark_ratio_max"] and row["entropy"] < cfg.get("entropy_near_mono_max", 3.0):
            reasons.append("near_mono_dark")

    if cfg.get("bright_ratio_max") is not None:
        if row["bright_ratio"] > cfg["bright_ratio_max"] and row["entropy"] < cfg.get("entropy_near_mono_max", 3.0):
            reasons.append("near_mono_bright")

    soft_count = int(final_soft_flags.loc[row.name]) if row.name in final_soft_flags.index else 0
    if soft_count > cfg.get("max_soft_flags", 2):
        reasons.append(f"too_many_soft_flags_{soft_count}")

    return ", ".join(reasons) if reasons else "not_removed"


removed_df = audit_df[~final_mask].copy()
removed_df["removal_reason"] = removed_df.apply(lambda row: removal_reasons(row, best_config), axis=1)

reason_counts = removed_df["removal_reason"].value_counts().reset_index()
reason_counts.columns = ["reason", "count"]
reason_counts["pct_of_removed"] = 100 * reason_counts["count"] / max(len(removed_df), 1)

display(reason_counts.round(2))

plt.figure(figsize=(10, 5))
sns.barplot(data=reason_counts.head(10), x="count", y="reason")
plt.title("Top Removal Reasons", fontweight="bold")
plt.xlabel("Number of removed images")
plt.ylabel("Reason")
savefig("07_removed_reasons.png")
plt.show()


def show_removed_examples_by_reason(reason, n=8):
    sub = removed_df[removed_df["removal_reason"].str.contains(reason, regex=False, na=False)]
    if len(sub) == 0:
        print("No examples for:", reason)
        return

    sub = sub.sample(min(n, len(sub)), random_state=SEED)
    cols = min(4, len(sub))
    rows = int(np.ceil(len(sub) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.3))

    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sub.iterrows()):
        try:
            img = Image.open(row["path"]).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "Error", ha="center", va="center")

        ax.axis("off")
        ax.set_title(
            f'{row["label_name"]}\nblur={row["blur_laplacian"]:.1f}, min={row["min_side"]}\nent={row["entropy"]:.2f}',
            fontsize=8,
        )

    for ax in axes[len(sub):]:
        ax.axis("off")

    plt.suptitle(f"Removed examples: {reason}", fontweight="bold")
    safe = reason.replace(" ", "_").replace("/", "_")
    savefig(f"removed_examples_{safe}.png")
    plt.show()


for reason in reason_counts["reason"].head(5):
    show_removed_examples_by_reason(reason, n=8)

#Block 15 — Final report-ready summary

In [ ]:
print("=" * 80)
print("SWEET SPOT THRESHOLD EXPERIMENT — FINAL SUMMARY")
print("=" * 80)

print(f"Selected preset       : {best_preset_name}")
print(f"Total audited images  : {len(audit_df):,}")
print(f"Clean images retained : {len(final_clean_df):,} ({100 * len(final_clean_df) / len(audit_df):.2f}%)")
print(f"Removed images        : {len(removed_df):,} ({100 * len(removed_df) / len(audit_df):.2f}%)")
print()
print(f"Baseline macro F1     : {baseline_result['f1_macro']:.4f}")
print(f"Selected macro F1     : {best_preset_row['f1_macro']:.4f}")
print(f"Selected accuracy     : {best_preset_row['accuracy']:.4f}")
print(f"Selected score        : {best_preset_row['score']:.4f}")
print()
print("Class distribution after cleaning:")
display(final_clean_df["label_name"].value_counts().to_frame("count"))

print("Recommended config:")
display(pd.DataFrame([best_config]).T.rename(columns={0: "value"}))

print("Files created:")
print("-", final_clean_path)
print("-", final_config_path)
print("-", SWEEP_PATH)
print("-", PRESET_PATH)
print("-", AUDIT_PATH)